In [ ]:
import sys, json, logging, pickle
from pathlib import Path
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier, CatBoostRegressor
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)
from sklearn.model_selection import GroupShuffleSplit

sys.path.insert(0, str(Path.cwd().parent))
from src.utils import load_labeled_pd, load_active_stressors, load_best_combination

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

PROJ_ROOT = Path.cwd().parent
DATA_DIR = PROJ_ROOT / 'data'
MODEL_DIR = DATA_DIR / 'models'
MODEL_DIR.mkdir(exist_ok=True)

In [2]:
CONFIG = {
    'SEED': 42, 'TEST_SIZE': 0.2, 'CALIBRATION_SIZE': 0.1,
    'CATBOOST_ITERATIONS': 500, 'CATBOOST_LEARNING_RATE': 0.05, 'CATBOOST_DEPTH': 6,
    'MIN_POSITIVES': 30
}

In [3]:
labeled_pd = load_labeled_pd(DATA_DIR)
active_stressors = load_active_stressors(DATA_DIR)
best_combo = load_best_combination(DATA_DIR)

In [4]:
# Load numeric-only features
X_list = []
for name in best_combo:
    df = pd.read_parquet(DATA_DIR / f"features_{name}.parquet").drop(columns=['organism'], errors='ignore')
    X_list.append(df)
X_full = pd.concat(X_list, axis=1)

# Save feature column order
with open(DATA_DIR / 'train_feature_columns.json', 'w') as f:
    json.dump(X_full.columns.tolist(), f)

groups = labeled_pd['organism']
gss = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'], random_state=CONFIG['SEED'])
train_idx, test_idx = next(gss.split(X_full, labeled_pd[active_stressors[0]], groups=groups))

# Save split
import json
with open(DATA_DIR / 'train_test_indices.json', 'w') as f:
    json.dump({'train_idx': train_idx.tolist(), 'test_idx': test_idx.tolist()}, f)

In [ ]:
def train_stressor(stressor):
    """Train CatBoost classifier, filtering to organisms with experiment data.

    Organisms with all-zero labels for a stressor had no RB-TnSeq experiments
    under that condition — their zeros are missing data, not true negatives.
    Filtering to orgs_with_data removes ~100K false-negative proteins and raises
    effective positive rates from ~0.6% to ~1.8% for metals like Zn.
    """
    y_full = labeled_pd[stressor]

    # Restrict to organisms that actually ran experiments for this stressor.
    orgs_with_data = labeled_pd.groupby('organism')[stressor].sum()
    orgs_with_data = orgs_with_data[orgs_with_data > 0].index
    mask = labeled_pd['organism'].isin(orgs_with_data)

    y = y_full[mask]
    X = X_full[mask]
    g = groups[mask]

    if y.sum() < CONFIG['MIN_POSITIVES']:
        log.warning(f"{stressor}: only {int(y.sum())} positives after org-filter; skipping.")
        return None

    # Per-stressor split on the filtered subset (not the global train_idx/test_idx)
    gss_s = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'],
                              random_state=CONFIG['SEED'])
    s_train_idx, s_test_idx = next(gss_s.split(X, y, groups=g))

    X_tr, X_te = X.iloc[s_train_idx], X.iloc[s_test_idx]
    y_tr, y_te = y.iloc[s_train_idx], y.iloc[s_test_idx]
    g_tr = g.iloc[s_train_idx]

    gss_cal = GroupShuffleSplit(n_splits=1, test_size=CONFIG['CALIBRATION_SIZE'],
                                random_state=CONFIG['SEED'])
    sub_idx, cal_idx = next(gss_cal.split(X_tr, y_tr, groups=g_tr))
    X_sub, X_cal = X_tr.iloc[sub_idx], X_tr.iloc[cal_idx]
    y_sub, y_cal = y_tr.iloc[sub_idx], y_tr.iloc[cal_idx]

    if y_sub.nunique() < 2:
        log.warning(f"{stressor}: training set has only one class; skipping.")
        return None

    model = CatBoostClassifier(
        iterations=CONFIG['CATBOOST_ITERATIONS'],
        learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
        depth=CONFIG['CATBOOST_DEPTH'],
        cat_features=None, eval_metric='AUC',
        auto_class_weights='Balanced',
        random_seed=CONFIG['SEED'], verbose=50)
    model.fit(X_sub, y_sub, verbose=50)

    y_proba = model.predict_proba(X_te)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    cal_proba = model.predict_proba(X_cal)[:, 1]
    if y_cal.nunique() < 2:
        log.warning(f"{stressor}: calibration set has only one class; skipping Platt.")
        y_proba_cal = y_proba
        platt = None
    else:
        platt = LogisticRegression()
        platt.fit(cal_proba.reshape(-1, 1), y_cal)
        y_proba_cal = platt.predict_proba(y_proba.reshape(-1, 1))[:, 1]

    metrics = {
        'AUC': roc_auc_score(y_te, y_proba),
        'Accuracy': accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall': recall_score(y_te, y_pred, zero_division=0),
        'F1': f1_score(y_te, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_te, y_pred),
        'Pos rate': y_te.mean(),
        'n_orgs_train': int(g_tr.nunique()),
        'n_orgs_test': int(g.iloc[s_test_idx].nunique()),
    }

    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_final.cbm"))
    if platt is not None:
        with open(MODEL_DIR / f"stressor_{stressor}_platt.pkl", 'wb') as f:
            pickle.dump(platt, f)

    pd.DataFrame({
        'y_test': y_te.values, 'y_proba': y_proba,
        'y_proba_cal': y_proba_cal, 'group': g.iloc[s_test_idx].values,
    }).to_parquet(MODEL_DIR / f"stressor_{stressor}_predictions.parquet")

    return metrics

In [6]:
all_metrics = {}
for s in active_stressors:
    log.info(f"Training {s}")
    m = train_stressor(s)
    if m: all_metrics[s] = m

pd.DataFrame(all_metrics).T.to_csv(DATA_DIR / 'final_model_performance.csv')
log.info("Training complete.")

2026-06-28 19:53:15,885 INFO Training Zn


0:	total: 76.5ms	remaining: 38.2s


50:	total: 1.18s	remaining: 10.4s


100:	total: 2.21s	remaining: 8.74s


150:	total: 3.28s	remaining: 7.59s


...

450:	total: 9.79s	remaining: 1.06s


499:	total: 10.7s	remaining: 0us


2026-06-28 20:01:58,495 INFO Training Biofilm


2026-06-28 20:01:58,976 WARNING Biofilm: training set has only one class; skipping.


2026-06-28 20:01:59,001 INFO Training complete.


### Regression training on continuous fitness scores (metals)

Requires `labeled_pd.parquet` to contain `{stressor}_fit` continuous columns,
produced when NB01 is re-run on Seaborg with the updated Cell 6 / Cell 12.

**Why regression for metals**: Binary labels collapse the full fitness distribution
to 0/1 at a -2.0 threshold, discarding the gradient between "strongly essential"
(fitness ≈ -5) and "slightly impaired" (fitness ≈ -1). Regression on the raw score
also sidesteps the class-imbalance problem: every tested protein contributes signal
regardless of whether it crosses the binary threshold.

**Evaluation**: Spearman ρ (rank correlation) is the primary metric — does the model
correctly rank metal resistance genes above housekeeping genes? AUC_from_ranking is
a secondary check: treating the top-ranked predictions as "positive" should recover
the known binary-positive genes.

In [ ]:
def train_stressor_regression(stressor):
    """Train CatBoost regressor on continuous RB-TnSeq fitness scores.

    Requires a '{stressor}_fit' column in labeled_pd (produced by NB01 re-run).
    Proteins with NaN fitness are excluded — they had no experiment for this stressor.
    """
    fit_col = f"{stressor}_fit"
    if fit_col not in labeled_pd.columns:
        log.warning(f"{stressor}: column '{fit_col}' absent — re-run NB01 on Seaborg first.")
        return None

    mask = labeled_pd[fit_col].notna()
    y = labeled_pd.loc[mask, fit_col].astype(float)
    X = X_full[mask]
    g = groups[mask]
    n_tested = int(mask.sum())

    if n_tested < CONFIG['MIN_POSITIVES'] * 10:
        log.warning(f"{stressor}: only {n_tested} proteins with data; skipping regression.")
        return None

    log.info(f"{stressor}: {n_tested} proteins with fitness data "
             f"({int((y < -2).sum())} binary positives)")

    gss_s = GroupShuffleSplit(n_splits=1, test_size=CONFIG['TEST_SIZE'],
                              random_state=CONFIG['SEED'])
    s_train_idx, s_test_idx = next(gss_s.split(X, y, groups=g))

    X_tr, X_te = X.iloc[s_train_idx], X.iloc[s_test_idx]
    y_tr, y_te = y.iloc[s_train_idx], y.iloc[s_test_idx]

    model = CatBoostRegressor(
        iterations=CONFIG['CATBOOST_ITERATIONS'],
        learning_rate=CONFIG['CATBOOST_LEARNING_RATE'],
        depth=CONFIG['CATBOOST_DEPTH'],
        loss_function='RMSE',
        random_seed=CONFIG['SEED'], verbose=50)
    model.fit(X_tr, y_tr, verbose=50)

    y_pred = model.predict(X_te)
    rho, pval = spearmanr(y_te, y_pred)
    rmse = float(np.sqrt(np.mean((y_te.values - y_pred) ** 2)))

    # Secondary metric: does ranking by predicted fitness recover binary positives?
    y_binary_te = (y_te < -2).astype(int)
    auc_from_ranking = float(roc_auc_score(y_binary_te, -y_pred)) if y_binary_te.sum() > 0 else None

    metrics = {
        'Spearman_rho': float(rho),
        'Spearman_pval': float(pval),
        'RMSE': rmse,
        'AUC_from_ranking': auc_from_ranking,
        'n_tested': n_tested,
        'n_binary_pos': int((y < -2).sum()),
        'pos_rate': float((y < -2).mean()),
    }

    model.save_model(str(MODEL_DIR / f"stressor_{stressor}_regression.cbm"))
    pd.DataFrame({
        'y_test': y_te.values, 'y_pred': y_pred,
        'group': g.iloc[s_test_idx].values,
    }).to_parquet(MODEL_DIR / f"stressor_{stressor}_reg_predictions.parquet")

    return metrics


METAL_STRESSORS = ['Zn', 'Cu', 'Cd', 'Co', 'Ni', 'Cr', 'As', 'Hg', 'Pb', 'Mn', 'Fe', 'Se', 'Ag', 'Al']
reg_metrics = {}
for s in METAL_STRESSORS:
    log.info(f"Regression: {s}")
    m = train_stressor_regression(s)
    if m:
        reg_metrics[s] = m
        log.info(f"  Spearman rho={m['Spearman_rho']:.3f} (p={m['Spearman_pval']:.2e})  "
                 f"AUC_from_ranking={m['AUC_from_ranking']}")

if reg_metrics:
    pd.DataFrame(reg_metrics).T.to_csv(DATA_DIR / 'regression_model_metrics.csv')
    log.info(f"Regression metrics saved for {len(reg_metrics)} metal stressors.")
else:
    log.info("No regression results — re-run NB01 on Seaborg to generate '{stressor}_fit' columns.")

In [7]:
# Tune decision thresholds per stressor to maximize F1 score
best_thresholds = {}

for stressor in active_stressors:
    pred_file = MODEL_DIR / f"stressor_{stressor}_predictions.parquet"
    if not pred_file.exists():
        continue
    
    preds_df = pd.read_parquet(pred_file)
    y_test = preds_df['y_test'].values
    y_proba = preds_df['y_proba'].values
    
    # Sweep thresholds from 0.01 to 0.99
    thresholds = np.linspace(0.01, 0.99, 99)
    f1_scores = []
    
    for thresh in thresholds:
        y_pred = (y_proba >= thresh).astype(int)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        f1_scores.append(f1)
    
    # Find threshold maximizing F1
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]
    
    best_thresholds[stressor] = float(best_threshold)
    log.info(f"{stressor}: best threshold={best_threshold:.3f}, F1={best_f1:.4f}")

# Save best thresholds
with open(DATA_DIR / 'best_thresholds.json', 'w') as f:
    json.dump(best_thresholds, f, indent=2)

log.info(f"Saved best thresholds for {len(best_thresholds)} stressors to data/best_thresholds.json")

2026-06-28 20:01:59,831 INFO Zn: best threshold=0.280, F1=0.0107


2026-06-28 20:02:00,547 INFO Cu: best threshold=0.690, F1=0.0565


2026-06-28 20:02:01,357 INFO Cd: best threshold=0.010, F1=0.0172


2026-06-28 20:02:02,035 INFO Co: best threshold=0.600, F1=0.0538


2026-06-28 20:02:02,917 INFO Ni: best threshold=0.620, F1=0.0398


...

2026-06-28 20:02:27,346 INFO Nitrogen limitation: best threshold=0.010, F1=0.0000


2026-06-28 20:02:28,085 INFO Iron limitation: best threshold=0.010, F1=0.0000


2026-06-28 20:02:28,925 INFO UV: best threshold=0.680, F1=0.2197


2026-06-28 20:02:29,542 INFO Anaerobic: best threshold=0.010, F1=0.0036


2026-06-28 20:02:29,544 INFO Saved best thresholds for 46 stressors to data/best_thresholds.json
